## Objectif du notebook

Ce notebook contient les analyses principales du volet B Data Analyst du projet Néovolt Grid+.

Les analyses couvrent :
- la consommation globale ;
- la consommation par zone ;
- la consommation par type de client ;
- la consommation par segment ;
- la consommation selon le type de chauffage ;
- les saisonnalités ;
- la relation entre météo et consommation ;
- les incidents réseau ;
- les profils horaires ;
- les premiers enseignements métier.

Les analyses s’appuient sur les fichiers nettoyés produits dans le notebook `02_nettoyage_preparation.ipynb`.

Le fichier `consommation_preparee.csv` est généré localement et n’est pas versionné dans GitHub car il est volumineux.

In [67]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

In [68]:
ROOT_DIR = Path.cwd()

# Si le notebook est exécuté depuis le dossier notebooks, on remonte à la racine du projet
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parents[1]

DATA_CLEANED_DIR = ROOT_DIR / "volet-b-data-analyst" / "data_cleaned"

DATA_CLEANED_DIR

WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-b-data-analyst/data_cleaned')

In [69]:
# Chargement des fichiers préparés

conso = pd.read_csv(DATA_CLEANED_DIR / "consommation_preparee.csv")
incidents = pd.read_csv(DATA_CLEANED_DIR / "incidents_preparees.csv")
releves_horaires = pd.read_csv(DATA_CLEANED_DIR / "releves_horaires_preparees.csv")
reclamations = pd.read_csv(DATA_CLEANED_DIR / "reclamations_preparees.csv")
fraudes = pd.read_csv(DATA_CLEANED_DIR / "cas_fraude_confirmes_preparees.csv")
rapport_nettoyage = pd.read_csv(DATA_CLEANED_DIR / "rapport_nettoyage.csv")

print("conso :", conso.shape)
print("incidents :", incidents.shape)
print("releves_horaires :", releves_horaires.shape)
print("reclamations :", reclamations.shape)
print("fraudes :", fraudes.shape)
print("rapport_nettoyage :", rapport_nettoyage.shape)

conso : (511700, 36)
incidents : (420, 10)
releves_horaires : (21600, 8)
reclamations : (3000, 9)
fraudes : (24, 4)
rapport_nettoyage : (7, 2)


In [70]:
# Conversion des dates après rechargement CSV

conso["date"] = pd.to_datetime(conso["date"], errors="coerce")
incidents["date"] = pd.to_datetime(incidents["date"], errors="coerce")
incidents["date_debut"] = pd.to_datetime(incidents["date_debut"], errors="coerce")
releves_horaires["date"] = pd.to_datetime(releves_horaires["date"], errors="coerce")
releves_horaires["horodatage"] = pd.to_datetime(releves_horaires["horodatage"], errors="coerce")
reclamations["date"] = pd.to_datetime(reclamations["date"], errors="coerce")
fraudes["date_detection"] = pd.to_datetime(fraudes["date_detection"], errors="coerce")

In [71]:
# Aperçu des colonnes du dataset principal

print("Nombre de lignes :", conso.shape[0])
print("Nombre de colonnes :", conso.shape[1])

conso.columns.tolist()

Nombre de lignes : 511700
Nombre de colonnes : 36


['id_pdl',
 'date',
 'consommation_kwh',
 'zone',
 'annee',
 'mois',
 'jour',
 'jour_semaine',
 'nom_jour',
 'weekend',
 'saison',
 'id_client',
 'type_client',
 'puissance_souscrite_kva',
 'type_chauffage',
 'type_compteur',
 'date_pose',
 'statut',
 'segment',
 'commune',
 'code_postal',
 'date_entree',
 'nb_personnes_foyer',
 'surface_m2',
 'temp_moyenne_c',
 'temp_min_c',
 'temp_max_c',
 'consommation_kwh_brute',
 'flag_conso_manquante',
 'flag_conso_negative',
 'flag_conso_aberrante_iqr',
 'consommation_kwh_clean',
 'flag_conso_imputee',
 'degres_jour_chauffage',
 'jour_froid',
 'jour_chaud']

## 1. Indicateurs globaux de consommation

Cette première analyse donne une vision générale du volume de données et de la consommation observée après nettoyage.

Les indicateurs portent sur :
- le nombre de points de livraison ;
- le nombre de clients ;
- la période couverte ;
- la consommation totale ;
- la consommation moyenne journalière ;
- la part de valeurs imputées.

In [72]:
# Indicateurs globaux

kpis_globaux = {
    "nb_lignes": len(conso),
    "nb_pdl": conso["id_pdl"].nunique(),
    "nb_clients": conso["id_client"].nunique(),
    "date_min": conso["date"].min(),
    "date_max": conso["date"].max(),
    "consommation_totale_kwh_clean": round(conso["consommation_kwh_clean"].sum(), 2),
    "consommation_moyenne_jour_kwh": round(conso["consommation_kwh_clean"].mean(), 2),
    "consommation_mediane_jour_kwh": round(conso["consommation_kwh_clean"].median(), 2),
    "part_lignes_imputees_%": round(conso["flag_conso_imputee"].mean() * 100, 2),
}

kpis_globaux_df = pd.DataFrame(
    list(kpis_globaux.items()),
    columns=["indicateur", "valeur"]
)

kpis_globaux_df

,indicateur,valeur
0,nb_lignes,511700
1,nb_pdl,700
2,nb_clients,700
3,date_min,2024-01-01 00:00:00
4,date_max,2025-12-31 00:00:00
5,consommation_totale_kwh_clean,12186856.03
6,consommation_moyenne_jour_kwh,23.82
7,consommation_mediane_jour_kwh,13.36
8,part_lignes_imputees_%,12.13


In [73]:
# Rapport de qualité issu du notebook de nettoyage

rapport_nettoyage

,indicateur,valeur
0,lignes_consommation_preparee,511700.00
1,consommations_manquantes_initiales,6838.00
2,consommations_negatives_detectees,1032.00
3,consommations_aberrantes_iqr_detectees,54179.00
4,consommations_imputees,62049.00
5,valeurs_manquantes_restantes_conso_clean,0.00
6,borne_haute_iqr_consommation,99.03


In [74]:
# Consommation mensuelle globale

conso_mensuelle = (
    conso
    .groupby(["annee", "mois"], as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        nb_releves=("consommation_kwh_clean", "count")
    )
)

conso_mensuelle["periode"] = pd.to_datetime(
    conso_mensuelle["annee"].astype(str) + "-" + conso_mensuelle["mois"].astype(str) + "-01"
)

conso_mensuelle.head()

,annee,mois,consommation_totale_kwh,consommation_moyenne_kwh,nb_releves,periode
0,2024,1,626755.020,28.882720,21700,2024-01-01
1,2024,2,573341.275,28.243413,20300,2024-02-01
2,2024,3,566391.855,26.101007,21700,2024-03-01
3,2024,4,500094.545,23.814026,21000,2024-04-01
4,2024,5,460200.220,21.207383,21700,2024-05-01


In [75]:
# Visualisation de la consommation mensuelle

fig = px.line(
    conso_mensuelle,
    x="periode",
    y="consommation_totale_kwh",
    title="Évolution mensuelle de la consommation totale nettoyée",
    labels={
        "periode": "Période",
        "consommation_totale_kwh": "Consommation totale (kWh)"
    }
)

fig.show()

Ce graphique permet d’observer l’évolution mensuelle de la consommation sur la période étudiée.

Les pics éventuels devront être rapprochés :
- des saisons ;
- des températures moyennes ;
- des types de clients ;
- des incidents réseau ;
- des périodes de forte demande énergétique.

## 2. Analyse des profils de consommation

Cette section analyse la consommation selon plusieurs dimensions métier :
- zone géographique ;
- type de client ;
- segment commercial ;
- type de chauffage ;
- saison.

L’objectif est d’identifier les profils de consommation les plus importants et les facteurs qui expliquent les variations observées.

In [76]:
# Consommation par zone

conso_par_zone = (
    conso
    .groupby("zone", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        nb_pdl=("id_pdl", "nunique"),
        nb_clients=("id_client", "nunique"),
        nb_releves=("consommation_kwh_clean", "count"),
        part_lignes_imputees=("flag_conso_imputee", "mean")
    )
)

conso_par_zone["part_lignes_imputees_%"] = round(conso_par_zone["part_lignes_imputees"] * 100, 2)
conso_par_zone = conso_par_zone.sort_values(by="consommation_totale_kwh", ascending=False)

conso_par_zone

,zone,consommation_totale_kwh,consommation_moyenne_kwh,nb_pdl,nb_clients,nb_releves,part_lignes_imputees,part_lignes_imputees_%
3,Parc-Tertiaire,2924597.025,40.412290,99,99,72369,0.132225,13.22
7,Zone-Industrielle,1880183.905,41.485016,62,62,45322,0.566833,56.68
1,Centre-Ville,1596861.085,21.628599,101,101,73831,0.125002,12.50
5,Rives-Sud,1448834.265,20.224382,98,98,71638,0.084173,8.42
0,Bourg-Ancien,1383393.480,24.262399,78,78,57018,0.114367,11.44
6,Val-Nord,1192318.730,16.643663,98,98,71638,0.024652,2.47
2,Coteaux-Ouest,984087.950,16.219537,83,83,60673,0.026750,2.67
4,Plateau-Est,776579.595,13.115462,81,81,59211,0.027377,2.74


In [77]:
fig = px.bar(
    conso_par_zone,
    x="zone",
    y="consommation_totale_kwh",
    title="Consommation totale nettoyée par zone",
    labels={
        "zone": "Zone",
        "consommation_totale_kwh": "Consommation totale (kWh)"
    }
)

fig.show()

Cette analyse permet d’identifier les zones qui concentrent le plus de consommation.  
Ces zones sont prioritaires pour l’exploitation réseau, car elles peuvent représenter un risque plus élevé lors des périodes de pointe.

Il faut cependant interpréter ces résultats avec prudence : une zone peut consommer davantage simplement parce qu’elle contient plus de points de livraison ou davantage de clients professionnels / industriels.

In [78]:
# Consommation par type de client

conso_par_type_client = (
    conso
    .groupby("type_client", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        consommation_mediane_kwh=("consommation_kwh_clean", "median"),
        nb_pdl=("id_pdl", "nunique"),
        nb_releves=("consommation_kwh_clean", "count")
    )
)

conso_par_type_client = conso_par_type_client.sort_values(
    by="consommation_totale_kwh",
    ascending=False
)

conso_par_type_client

,type_client,consommation_totale_kwh,consommation_moyenne_kwh,consommation_mediane_kwh,nb_pdl,nb_releves
1,professionnel,7345927.150,47.626294,46.31,211,154241
2,residentiel,3376819.885,10.843786,9.32,426,311406
0,industriel,1464109.000,31.791827,34.75,63,46053


In [79]:
fig = px.bar(
    conso_par_type_client,
    x="type_client",
    y="consommation_moyenne_kwh",
    title="Consommation moyenne journalière par type de client",
    labels={
        "type_client": "Type de client",
        "consommation_moyenne_kwh": "Consommation moyenne journalière (kWh)"
    }
)

fig.show()

Cette analyse permet de comparer les comportements de consommation selon les catégories d’usage.  
Les clients industriels ou professionnels peuvent avoir une consommation moyenne beaucoup plus élevée que les clients résidentiels.

Pour le pilotage métier, cette distinction est importante :
- l’exploitation réseau surveille les profils fortement consommateurs ;
- la direction financière peut estimer la contribution de chaque segment ;
- la relation client peut adapter les messages et services selon les usages.

In [80]:
# Consommation par segment commercial

conso_par_segment = (
    conso
    .groupby("segment", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        consommation_mediane_kwh=("consommation_kwh_clean", "median"),
        nb_clients=("id_client", "nunique"),
        nb_pdl=("id_pdl", "nunique")
    )
)

conso_par_segment = conso_par_segment.sort_values(
    by="consommation_totale_kwh",
    ascending=False
)

conso_par_segment

,segment,consommation_totale_kwh,consommation_moyenne_kwh,consommation_mediane_kwh,nb_clients,nb_pdl
3,petit_pro,4596311.650,47.997741,46.930,131,131
1,entreprise,3640174.050,41.154696,39.110,121,121
2,particulier,3376819.885,10.843786,9.320,426,426
0,collectivite,573550.450,35.664124,36.365,22,22


In [81]:
fig = px.bar(
    conso_par_segment,
    x="segment",
    y="consommation_totale_kwh",
    title="Consommation totale nettoyée par segment commercial",
    labels={
        "segment": "Segment",
        "consommation_totale_kwh": "Consommation totale (kWh)"
    }
)

fig.show()

In [82]:
# Consommation par type de chauffage

conso_par_chauffage = (
    conso
    .groupby("type_chauffage", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        consommation_mediane_kwh=("consommation_kwh_clean", "median"),
        nb_pdl=("id_pdl", "nunique")
    )
)

conso_par_chauffage = conso_par_chauffage.sort_values(
    by="consommation_moyenne_kwh",
    ascending=False
)

conso_par_chauffage

,type_chauffage,consommation_totale_kwh,consommation_moyenne_kwh,consommation_mediane_kwh,nb_pdl
3,reseau_chaleur,425845.790,38.836825,39.125,15
2,gaz,4591333.315,27.308234,15.095,230
1,electrique,4628690.090,22.295765,14.810,284
0,autre,2540986.840,20.327732,9.790,171


In [83]:
fig = px.bar(
    conso_par_chauffage,
    x="type_chauffage",
    y="consommation_moyenne_kwh",
    title="Consommation moyenne journalière selon le type de chauffage",
    labels={
        "type_chauffage": "Type de chauffage",
        "consommation_moyenne_kwh": "Consommation moyenne journalière (kWh)"
    }
)

fig.show()

Le type de chauffage est une variable importante pour expliquer les variations de consommation.  
Les clients chauffés à l’électricité sont susceptibles d’avoir une consommation plus sensible aux températures froides.

Cette information sera utile pour l’analyse météo et saisonnalité.

In [84]:
# Consommation par saison

ordre_saisons = ["hiver", "printemps", "ete", "automne"]

conso_par_saison = (
    conso
    .groupby("saison", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        nb_releves=("consommation_kwh_clean", "count")
    )
)

conso_par_saison["saison"] = pd.Categorical(
    conso_par_saison["saison"],
    categories=ordre_saisons,
    ordered=True
)

conso_par_saison = conso_par_saison.sort_values("saison")

conso_par_saison

,saison,consommation_totale_kwh,consommation_moyenne_kwh,nb_releves
2,hiver,3580219.485,28.257454,126700
3,printemps,3057774.860,23.740488,128800
1,ete,2608231.450,20.250244,128800
0,automne,2940630.240,23.081870,127400


In [85]:
fig = px.bar(
    conso_par_saison,
    x="saison",
    y="consommation_moyenne_kwh",
    title="Consommation moyenne journalière par saison",
    labels={
        "saison": "Saison",
        "consommation_moyenne_kwh": "Consommation moyenne journalière (kWh)"
    }
)

fig.show()

Cette analyse met en évidence la saisonnalité de la consommation.  
Une consommation plus élevée en hiver peut s’expliquer par les besoins de chauffage, en particulier pour les clients équipés en chauffage électrique.

Cette saisonnalité est importante pour Néovolt car elle influence :
- l’anticipation des pics ;
- les achats d’énergie ;
- la stabilité du réseau ;
- la préparation des périodes critiques.

In [86]:
# Analyse croisée : zone et type de client

conso_zone_type = (
    conso
    .groupby(["zone", "type_client"], as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        nb_pdl=("id_pdl", "nunique")
    )
)

conso_zone_type.head()

,zone,type_client,consommation_totale_kwh,consommation_moyenne_kwh,nb_pdl
0,Bourg-Ancien,industriel,97393.520,22.205545,6
1,Bourg-Ancien,professionnel,866163.575,53.859195,22
2,Bourg-Ancien,residentiel,419836.385,11.486632,50
3,Centre-Ville,industriel,116194.810,15.895323,10
4,Centre-Ville,professionnel,1038871.290,44.411392,32


In [87]:
fig = px.bar(
    conso_zone_type,
    x="zone",
    y="consommation_totale_kwh",
    color="type_client",
    title="Consommation totale par zone et type de client",
    labels={
        "zone": "Zone",
        "consommation_totale_kwh": "Consommation totale (kWh)",
        "type_client": "Type de client"
    }
)

fig.show()

Cette analyse croisée permet d’éviter une lecture trop globale des zones.  
Une zone fortement consommatrice peut être liée à une concentration de clients professionnels ou industriels.

Elle permet aussi d’orienter les tableaux de bord :
- vue exploitation : zones à surveiller ;
- vue finance : segments à fort poids économique ;
- vue relation client : profils nécessitant un accompagnement spécifique.

## 3. Analyse temporelle et influence de la météo

Cette section analyse la consommation dans le temps et son lien avec les conditions météorologiques.

L’objectif est d’identifier :
- les périodes de forte consommation ;
- les saisonnalités ;
- l’effet des températures froides ou chaudes ;
- le rôle des degrés-jour de chauffage ;
- les périodes à surveiller pour l’exploitation réseau.

In [88]:
# Agrégation de la consommation par jour

conso_journaliere = (
    conso
    .groupby("date", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        temp_moyenne_c=("temp_moyenne_c", "mean"),
        degres_jour_chauffage=("degres_jour_chauffage", "mean"),
        nb_releves=("consommation_kwh_clean", "count")
    )
)

conso_journaliere.head()

,date,consommation_totale_kwh,consommation_moyenne_kwh,temp_moyenne_c,degres_jour_chauffage,nb_releves
0,2024-01-01,21155.675,30.222393,3.889143,13.110857,700
1,2024-01-02,21209.150,30.298786,4.060386,12.939614,700
2,2024-01-03,20949.730,29.928186,5.193214,11.806786,700
3,2024-01-04,21531.630,30.759471,2.804043,14.195957,700
4,2024-01-05,21302.615,30.432307,2.830414,14.169586,700


In [89]:
fig = px.line(
    conso_journaliere,
    x="date",
    y="consommation_totale_kwh",
    title="Évolution journalière de la consommation totale nettoyée",
    labels={
        "date": "Date",
        "consommation_totale_kwh": "Consommation totale journalière (kWh)"
    }
)

fig.show()

La consommation journalière permet d’identifier les périodes de forte demande.  
Les pics observés doivent être rapprochés des conditions météo, des saisons et des incidents réseau.

Pour Néovolt, cette lecture est utile pour :
- anticiper les achats d’énergie ;
- préparer les périodes de pointe ;
- surveiller les zones sensibles ;
- mieux piloter l’exploitation réseau.

In [90]:
# Relation entre température moyenne et consommation totale journalière

fig = px.scatter(
    conso_journaliere,
    x="temp_moyenne_c",
    y="consommation_totale_kwh",
    title="Relation entre température moyenne et consommation journalière",
    labels={
        "temp_moyenne_c": "Température moyenne (°C)",
        "consommation_totale_kwh": "Consommation totale journalière (kWh)"
    },
    trendline="ols"
)

fig.show()

In [91]:
# Corrélations entre météo et consommation

correlations_meteo = conso_journaliere[
    [
        "consommation_totale_kwh",
        "temp_moyenne_c",
        "degres_jour_chauffage"
    ]
].corr()

correlations_meteo

,consommation_totale_kwh,temp_moyenne_c,degres_jour_chauffage
consommation_totale_kwh,1.000000,-0.738746,0.747613
temp_moyenne_c,-0.738746,1.000000,-0.983843
degres_jour_chauffage,0.747613,-0.983843,1.000000


Une corrélation négative entre température moyenne et consommation signifie généralement que la consommation augmente lorsque la température baisse.  
Cela est cohérent avec les besoins de chauffage, notamment pour les clients chauffés à l’électricité.

Les degrés-jour de chauffage permettent de mieux mesurer l’intensité du froid et son impact sur la consommation.

In [92]:
fig = px.scatter(
    conso_journaliere,
    x="degres_jour_chauffage",
    y="consommation_totale_kwh",
    title="Relation entre degrés-jour de chauffage et consommation journalière",
    labels={
        "degres_jour_chauffage": "Degrés-jour de chauffage",
        "consommation_totale_kwh": "Consommation totale journalière (kWh)"
    },
    trendline="ols"
)

fig.show()

In [93]:
# Classification simple des journées selon la température

def classer_jour_meteo(temp):
    if pd.isna(temp):
        return "temperature_inconnue"
    elif temp < 7:
        return "jour_froid"
    elif temp > 25:
        return "jour_chaud"
    else:
        return "jour_normal"

conso_journaliere["classe_meteo"] = conso_journaliere["temp_moyenne_c"].apply(classer_jour_meteo)

conso_par_classe_meteo = (
    conso_journaliere
    .groupby("classe_meteo", as_index=False)
    .agg(
        consommation_moyenne_jour_kwh=("consommation_totale_kwh", "mean"),
        consommation_mediane_jour_kwh=("consommation_totale_kwh", "median"),
        nb_jours=("date", "count")
    )
)

conso_par_classe_meteo = conso_par_classe_meteo.sort_values(
    by="consommation_moyenne_jour_kwh",
    ascending=False
)

conso_par_classe_meteo

,classe_meteo,consommation_moyenne_jour_kwh,consommation_mediane_jour_kwh,nb_jours
0,jour_froid,19662.094726,20627.8450,201
1,jour_normal,15537.311311,15542.1825,530


In [94]:
fig = px.bar(
    conso_par_classe_meteo,
    x="classe_meteo",
    y="consommation_moyenne_jour_kwh",
    title="Consommation moyenne journalière selon la classe météo",
    labels={
        "classe_meteo": "Classe météo",
        "consommation_moyenne_jour_kwh": "Consommation moyenne journalière (kWh)"
    }
)

fig.show()

In [95]:
# Agrégation mensuelle avec température moyenne

conso_mensuelle_meteo = (
    conso
    .groupby(["annee", "mois"], as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        temp_moyenne_c=("temp_moyenne_c", "mean"),
        degres_jour_chauffage=("degres_jour_chauffage", "mean")
    )
)

conso_mensuelle_meteo["periode"] = pd.to_datetime(
    conso_mensuelle_meteo["annee"].astype(str) + "-"
    + conso_mensuelle_meteo["mois"].astype(str) + "-01"
)

conso_mensuelle_meteo.head()

,annee,mois,consommation_totale_kwh,consommation_moyenne_kwh,temp_moyenne_c,degres_jour_chauffage,periode
0,2024,1,626755.020,28.882720,3.425959,13.574041,2024-01-01
1,2024,2,573341.275,28.243413,4.403022,12.596978,2024-02-01
2,2024,3,566391.855,26.101007,7.306348,9.693652,2024-03-01
3,2024,4,500094.545,23.814026,11.857221,5.157233,2024-04-01
4,2024,5,460200.220,21.207383,16.562511,1.297673,2024-05-01


In [96]:
fig = px.line(
    conso_mensuelle_meteo,
    x="periode",
    y=["consommation_totale_kwh", "temp_moyenne_c"],
    title="Consommation mensuelle et température moyenne",
    labels={
        "periode": "Période",
        "value": "Valeur",
        "variable": "Indicateur"
    }
)

fig.show()

In [97]:
# Analyse croisée saison et météo

conso_saison_meteo = (
    conso
    .groupby("saison", as_index=False)
    .agg(
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        temp_moyenne_c=("temp_moyenne_c", "mean"),
        degres_jour_chauffage=("degres_jour_chauffage", "mean"),
        nb_releves=("consommation_kwh_clean", "count")
    )
)

conso_saison_meteo["saison"] = pd.Categorical(
    conso_saison_meteo["saison"],
    categories=["hiver", "printemps", "ete", "automne"],
    ordered=True
)

conso_saison_meteo = conso_saison_meteo.sort_values("saison")

conso_saison_meteo

,saison,consommation_moyenne_kwh,consommation_totale_kwh,temp_moyenne_c,degres_jour_chauffage,nb_releves
2,hiver,28.257454,3580219.485,4.414440,12.585560,126700
3,printemps,23.740488,3057774.860,11.965416,5.339634,128800
1,ete,20.250244,2608231.450,20.460213,0.131999,128800
0,automne,23.081870,2940630.240,13.000896,4.461240,127400


In [98]:
fig = px.bar(
    conso_saison_meteo,
    x="saison",
    y="consommation_moyenne_kwh",
    title="Consommation moyenne et saisonnalité",
    labels={
        "saison": "Saison",
        "consommation_moyenne_kwh": "Consommation moyenne journalière (kWh)"
    }
)

fig.show()

## 4. Analyse des incidents réseau

Cette section analyse les incidents réseau afin d’identifier :
- les zones les plus touchées ;
- les types d’incidents les plus fréquents ;
- les causes principales ;
- les volumes de points de livraison impactés ;
- le lien éventuel entre incidents et consommation.

In [99]:
# Aperçu du fichier incidents

print("Dimensions incidents :", incidents.shape)
incidents.head()
incidents.columns.tolist()

Dimensions incidents : (420, 10)


['id_incident',
 'date_debut',
 'duree_minutes',
 'zone',
 'type',
 'nb_pdl_impactes',
 'cause',
 'date',
 'annee',
 'mois']

In [100]:
# Indicateurs globaux sur les incidents réseau

kpis_incidents = {
    "nb_incidents": len(incidents),
    "nb_zones_touchees": incidents["zone"].nunique(),
    "date_min": incidents["date"].min(),
    "date_max": incidents["date"].max(),
    "duree_totale_minutes": incidents["duree_minutes"].sum(),
    "duree_moyenne_minutes": round(incidents["duree_minutes"].mean(), 2),
    "nb_pdl_impactes_total": incidents["nb_pdl_impactes"].sum(),
    "nb_pdl_impactes_moyen": round(incidents["nb_pdl_impactes"].mean(), 2),
}

kpis_incidents_df = pd.DataFrame(
    list(kpis_incidents.items()),
    columns=["indicateur", "valeur"]
)

kpis_incidents_df

,indicateur,valeur
0,nb_incidents,420
1,nb_zones_touchees,8
2,date_min,2024-01-01 00:00:00
3,date_max,2025-12-28 00:00:00
4,duree_totale_minutes,41313
5,duree_moyenne_minutes,98.36
6,nb_pdl_impactes_total,209120
7,nb_pdl_impactes_moyen,497.9


In [101]:
# Incidents par zone

incidents_par_zone = (
    incidents
    .groupby("zone", as_index=False)
    .agg(
        nb_incidents=("id_incident", "count"),
        duree_totale_minutes=("duree_minutes", "sum"),
        duree_moyenne_minutes=("duree_minutes", "mean"),
        nb_pdl_impactes_total=("nb_pdl_impactes", "sum"),
        nb_pdl_impactes_moyen=("nb_pdl_impactes", "mean")
    )
)

incidents_par_zone["duree_moyenne_minutes"] = incidents_par_zone["duree_moyenne_minutes"].round(2)
incidents_par_zone["nb_pdl_impactes_moyen"] = incidents_par_zone["nb_pdl_impactes_moyen"].round(2)

incidents_par_zone = incidents_par_zone.sort_values(
    by="nb_incidents",
    ascending=False
)

incidents_par_zone

,zone,nb_incidents,duree_totale_minutes,duree_moyenne_minutes,nb_pdl_impactes_total,nb_pdl_impactes_moyen
0,Bourg-Ancien,69,6433,93.23,31680,459.13
6,Val-Nord,62,5969,96.27,31947,515.27
7,Zone-Industrielle,52,4819,92.67,26032,500.62
3,Parc-Tertiaire,51,4844,94.98,24896,488.16
2,Coteaux-Ouest,50,5208,104.16,23772,475.44
5,Rives-Sud,48,4196,87.42,24900,518.75
1,Centre-Ville,45,4699,104.42,23011,511.36
4,Plateau-Est,43,5145,119.65,22882,532.14


In [102]:
fig = px.bar(
    incidents_par_zone,
    x="zone",
    y="nb_incidents",
    title="Nombre d’incidents réseau par zone",
    labels={
        "zone": "Zone",
        "nb_incidents": "Nombre d’incidents"
    }
)

fig.show()

Cette vue permet d’identifier les zones les plus exposées aux incidents réseau.

Pour l’exploitation réseau, ces zones peuvent devenir prioritaires pour :
- renforcer la surveillance ;
- planifier des opérations de maintenance ;
- croiser les incidents avec les pics de consommation ;
- améliorer la qualité de service client.

In [103]:
# Incidents par type

incidents_par_type = (
    incidents
    .groupby("type", as_index=False)
    .agg(
        nb_incidents=("id_incident", "count"),
        duree_totale_minutes=("duree_minutes", "sum"),
        nb_pdl_impactes_total=("nb_pdl_impactes", "sum")
    )
)

incidents_par_type = incidents_par_type.sort_values(
    by="nb_incidents",
    ascending=False
)

incidents_par_type

,type,nb_incidents,duree_totale_minutes,nb_pdl_impactes_total
4,surtension,103,10326,58385
2,maintenance_programmee,91,8215,10078
0,baisse_tension,81,8064,48604
1,coupure,76,7727,47572
3,panne_poste,69,6981,44481


In [104]:
fig = px.bar(
    incidents_par_type,
    x="type",
    y="nb_incidents",
    title="Nombre d’incidents par type",
    labels={
        "type": "Type d’incident",
        "nb_incidents": "Nombre d’incidents"
    }
)

fig.show()

In [105]:
# Incidents par cause

incidents_par_cause = (
    incidents
    .groupby("cause", as_index=False)
    .agg(
        nb_incidents=("id_incident", "count"),
        duree_totale_minutes=("duree_minutes", "sum"),
        nb_pdl_impactes_total=("nb_pdl_impactes", "sum")
    )
)

incidents_par_cause = incidents_par_cause.sort_values(
    by="nb_incidents",
    ascending=False
)

incidents_par_cause

,cause,nb_incidents,duree_totale_minutes,nb_pdl_impactes_total
1,inconnue,84,7270,37357
5,vetuste_materiel,80,7196,41673
2,intemperie,74,7515,36337
0,defaut_isolement,65,7292,35887
3,surcharge,60,6017,29420
4,travaux_tiers,57,6023,28446


In [106]:
fig = px.bar(
    incidents_par_cause,
    x="cause",
    y="nb_incidents",
    title="Nombre d’incidents par cause",
    labels={
        "cause": "Cause",
        "nb_incidents": "Nombre d’incidents"
    }
)

fig.show()

L’analyse par cause permet d’orienter les actions correctives.

Par exemple :
- une part importante liée aux intempéries peut justifier une meilleure anticipation météo ;
- une part liée à la vétusté du matériel peut orienter les investissements réseau ;
- une part liée aux surcharges peut être rapprochée des pics de consommation.

In [107]:
# Agrégation consommation par date et zone

conso_zone_jour = (
    conso
    .groupby(["date", "zone"], as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        nb_pdl=("id_pdl", "nunique")
    )
)

conso_zone_jour.head()

,date,zone,consommation_totale_kwh,consommation_moyenne_kwh,nb_pdl
0,2024-01-01,Bourg-Ancien,2343.510,30.045000,78
1,2024-01-01,Centre-Ville,2654.935,26.286485,101
2,2024-01-01,Coteaux-Ouest,1853.305,22.328976,83
3,2024-01-01,Parc-Tertiaire,5064.295,51.154495,99
4,2024-01-01,Plateau-Est,1217.180,15.026914,81


In [108]:
# Agrégation incidents par date et zone

incidents_zone_jour = (
    incidents
    .groupby(["date", "zone"], as_index=False)
    .agg(
        nb_incidents=("id_incident", "count"),
        duree_totale_minutes=("duree_minutes", "sum"),
        nb_pdl_impactes_total=("nb_pdl_impactes", "sum")
    )
)

incidents_zone_jour.head()

,date,zone,nb_incidents,duree_totale_minutes,nb_pdl_impactes_total
0,2024-01-01,Plateau-Est,1,261,256
1,2024-01-03,Plateau-Est,1,185,283
2,2024-01-04,Centre-Ville,1,151,125
3,2024-01-04,Parc-Tertiaire,1,155,419
4,2024-01-08,Plateau-Est,1,140,133


In [109]:
# Fusion consommation + incidents

conso_incidents = conso_zone_jour.merge(
    incidents_zone_jour,
    on=["date", "zone"],
    how="left"
)

conso_incidents["nb_incidents"] = conso_incidents["nb_incidents"].fillna(0)
conso_incidents["duree_totale_minutes"] = conso_incidents["duree_totale_minutes"].fillna(0)
conso_incidents["nb_pdl_impactes_total"] = conso_incidents["nb_pdl_impactes_total"].fillna(0)

conso_incidents["jour_avec_incident"] = conso_incidents["nb_incidents"] > 0

conso_incidents.head()

,date,zone,consommation_totale_kwh,consommation_moyenne_kwh,nb_pdl,nb_incidents,duree_totale_minutes,nb_pdl_impactes_total,jour_avec_incident
0,2024-01-01,Bourg-Ancien,2343.510,30.045000,78,0.0,0.0,0.0,False
1,2024-01-01,Centre-Ville,2654.935,26.286485,101,0.0,0.0,0.0,False
2,2024-01-01,Coteaux-Ouest,1853.305,22.328976,83,0.0,0.0,0.0,False
3,2024-01-01,Parc-Tertiaire,5064.295,51.154495,99,0.0,0.0,0.0,False
4,2024-01-01,Plateau-Est,1217.180,15.026914,81,1.0,261.0,256.0,True


In [110]:
# Comparaison consommation des jours avec / sans incident

comparaison_incidents = (
    conso_incidents
    .groupby("jour_avec_incident", as_index=False)
    .agg(
        consommation_moyenne_zone_jour_kwh=("consommation_totale_kwh", "mean"),
        consommation_mediane_zone_jour_kwh=("consommation_totale_kwh", "median"),
        nb_lignes=("date", "count"),
        nb_incidents_total=("nb_incidents", "sum"),
        nb_pdl_impactes_total=("nb_pdl_impactes_total", "sum")
    )
)

comparaison_incidents

,jour_avec_incident,consommation_moyenne_zone_jour_kwh,consommation_mediane_zone_jour_kwh,nb_lignes,nb_incidents_total,nb_pdl_impactes_total
0,False,2081.468811,1880.420,5445,0.0,0.0
1,True,2117.266402,1886.255,403,420.0,209120.0


In [111]:
fig = px.bar(
    comparaison_incidents,
    x="jour_avec_incident",
    y="consommation_moyenne_zone_jour_kwh",
    title="Consommation moyenne zone/jour selon la présence d’un incident",
    labels={
        "jour_avec_incident": "Jour avec incident",
        "consommation_moyenne_zone_jour_kwh": "Consommation moyenne zone/jour (kWh)"
    }
)

fig.show()

Cette comparaison permet de voir si les jours avec incidents correspondent à des niveaux de consommation différents.
NB: Un incident peut être causé par une surcharge, mais il peut aussi être lié à une panne matérielle, une intempérie ou une maintenance programmée.


In [112]:
# Top 10 des couples date/zone les plus impactés

top_incidents_zone_jour = (
    conso_incidents
    .sort_values(
        by=["nb_pdl_impactes_total", "nb_incidents"],
        ascending=False
    )
    .head(10)
)

top_incidents_zone_jour

,date,zone,consommation_totale_kwh,consommation_moyenne_kwh,nb_pdl,nb_incidents,duree_totale_minutes,nb_pdl_impactes_total,jour_avec_incident
1165,2024-05-25,Rives-Sud,1222.605,12.475561,98,2.0,218.0,1969.0,True
4222,2025-06-11,Val-Nord,1394.000,14.224490,98,2.0,84.0,1729.0,True
1218,2024-06-01,Coteaux-Ouest,972.205,11.713313,83,2.0,129.0,1716.0,True
4751,2025-08-16,Zone-Industrielle,1816.185,29.293306,62,2.0,10.0,1502.0,True
2448,2024-11-02,Bourg-Ancien,1422.260,18.234103,78,2.0,257.0,1469.0,True
2311,2024-10-15,Zone-Industrielle,2706.900,43.659677,62,2.0,127.0,1264.0,True
723,2024-03-31,Parc-Tertiaire,3108.580,31.399798,99,1.0,217.0,1193.0,True
976,2024-05-02,Bourg-Ancien,1963.900,25.178205,78,2.0,203.0,1192.0,True
5336,2025-10-29,Bourg-Ancien,2022.735,25.932500,78,1.0,5.0,1191.0,True
1673,2024-07-28,Centre-Ville,1505.125,14.902228,101,1.0,52.0,1183.0,True


## 5. Analyse des profils horaires

Cette section exploite le fichier `releves_horaires_echantillon.csv`, préparé dans le notebook de nettoyage.

L’objectif est d’analyser les courbes de charge horaires afin d’identifier :
- les heures de pointe ;
- les différences entre jours ouvrés et week-ends ;
- les profils moyens de consommation par heure ;
- les zones les plus consommatrices sur l’échantillon horaire.


In [113]:
# Aperçu des relevés horaires préparés

print("Dimensions relevés horaires :", releves_horaires.shape)
releves_horaires.head()

releves_horaires.columns.tolist()

Dimensions relevés horaires : (21600, 8)


['id_pdl',
 'horodatage',
 'consommation_kwh',
 'zone',
 'date',
 'heure',
 'jour_semaine',
 'weekend']

In [114]:
# Indicateurs globaux sur l'échantillon horaire

kpis_horaires = {
    "nb_lignes": len(releves_horaires),
    "nb_pdl": releves_horaires["id_pdl"].nunique(),
    "nb_zones": releves_horaires["zone"].nunique(),
    "date_min": releves_horaires["horodatage"].min(),
    "date_max": releves_horaires["horodatage"].max(),
    "consommation_totale_kwh": round(releves_horaires["consommation_kwh"].sum(), 2),
    "consommation_moyenne_horaire_kwh": round(releves_horaires["consommation_kwh"].mean(), 2),
    "consommation_max_horaire_kwh": round(releves_horaires["consommation_kwh"].max(), 2)
}

kpis_horaires_df = pd.DataFrame(
    list(kpis_horaires.items()),
    columns=["indicateur", "valeur"]
)

kpis_horaires_df

,indicateur,valeur
0,nb_lignes,21600
1,nb_pdl,30
2,nb_zones,8
3,date_min,2025-08-23 00:00:00
4,date_max,2025-09-21 23:00:00
5,consommation_totale_kwh,68711.82
6,consommation_moyenne_horaire_kwh,3.18
7,consommation_max_horaire_kwh,63.06


In [115]:
# Profil moyen de consommation par heure

profil_horaire = (
    releves_horaires
    .groupby("heure", as_index=False)
    .agg(
        consommation_moyenne_kwh=("consommation_kwh", "mean"),
        consommation_mediane_kwh=("consommation_kwh", "median"),
        consommation_totale_kwh=("consommation_kwh", "sum"),
        nb_releves=("consommation_kwh", "count")
    )
)

profil_horaire = profil_horaire.sort_values("heure")

profil_horaire

,heure,consommation_moyenne_kwh,consommation_mediane_kwh,consommation_totale_kwh,nb_releves
0,0,0.794011,0.210,714.61,900
1,1,0.591333,0.160,532.20,900
2,2,0.599589,0.170,539.63,900
3,3,0.606933,0.170,546.24,900
4,4,0.600844,0.160,540.76,900
5,5,0.591011,0.160,531.91,900
6,6,2.491322,0.570,2242.19,900
7,7,5.864067,0.580,5277.66,900
8,8,5.941522,0.580,5347.37,900
9,9,5.802822,0.580,5222.54,900


In [116]:
fig = px.line(
    profil_horaire,
    x="heure",
    y="consommation_moyenne_kwh",
    markers=True,
    title="Profil moyen de consommation par heure",
    labels={
        "heure": "Heure de la journée",
        "consommation_moyenne_kwh": "Consommation moyenne horaire (kWh)"
    }
)

fig.show()

Le profil moyen par heure permet d’identifier les heures où la consommation est la plus élevée.

Ces heures de pointe sont importantes pour l’exploitation réseau, car elles peuvent nécessiter :
- une surveillance renforcée ;
- une meilleure anticipation de la charge ;
- des actions d’effacement ou de sensibilisation client ;
- une meilleure planification des achats d’énergie.

In [117]:
# Identification des heures de pointe

seuil_pointe = profil_horaire["consommation_moyenne_kwh"].quantile(0.75)

profil_horaire["heure_de_pointe"] = profil_horaire["consommation_moyenne_kwh"] >= seuil_pointe

heures_de_pointe = profil_horaire[profil_horaire["heure_de_pointe"]].sort_values(
    by="consommation_moyenne_kwh",
    ascending=False
)

heures_de_pointe

,heure,consommation_moyenne_kwh,consommation_mediane_kwh,consommation_totale_kwh,nb_releves,heure_de_pointe
18,18,6.832800,0.66,6149.52,900,True
20,20,6.810656,0.68,6129.59,900,True
19,19,6.790089,0.67,6111.08,900,True
17,17,6.742778,0.66,6068.50,900,True
8,8,5.941522,0.58,5347.37,900,True
7,7,5.864067,0.58,5277.66,900,True


In [118]:
# Profil horaire selon jour ouvré ou week-end

profil_horaire_weekend = (
    releves_horaires
    .groupby(["heure", "weekend"], as_index=False)
    .agg(
        consommation_moyenne_kwh=("consommation_kwh", "mean"),
        consommation_totale_kwh=("consommation_kwh", "sum"),
        nb_releves=("consommation_kwh", "count")
    )
)

profil_horaire_weekend["type_jour"] = np.where(
    profil_horaire_weekend["weekend"],
    "week-end",
    "jour_ouvre"
)

profil_horaire_weekend.head()

,heure,weekend,consommation_moyenne_kwh,consommation_totale_kwh,nb_releves,type_jour
0,0,False,0.794050,476.43,600,jour_ouvre
1,0,True,0.793933,238.18,300,week-end
2,1,False,0.595317,357.19,600,jour_ouvre
3,1,True,0.583367,175.01,300,week-end
4,2,False,0.601883,361.13,600,jour_ouvre


In [119]:
fig = px.line(
    profil_horaire_weekend,
    x="heure",
    y="consommation_moyenne_kwh",
    color="type_jour",
    markers=True,
    title="Profil horaire moyen : jours ouvrés vs week-ends",
    labels={
        "heure": "Heure de la journée",
        "consommation_moyenne_kwh": "Consommation moyenne horaire (kWh)",
        "type_jour": "Type de jour"
    }
)

fig.show()

In [120]:
# Consommation horaire moyenne par zone

profil_horaire_zone = (
    releves_horaires
    .groupby(["heure", "zone"], as_index=False)
    .agg(
        consommation_moyenne_kwh=("consommation_kwh", "mean"),
        consommation_totale_kwh=("consommation_kwh", "sum"),
        nb_pdl=("id_pdl", "nunique")
    )
)

profil_horaire_zone.head()

,heure,zone,consommation_moyenne_kwh,consommation_totale_kwh,nb_pdl
0,0,Bourg-Ancien,0.479667,14.39,1
1,0,Centre-Ville,0.143444,12.91,3
2,0,Coteaux-Ouest,0.211667,31.75,5
3,0,Parc-Tertiaire,1.671333,200.56,4
4,0,Plateau-Est,0.187083,22.45,4


In [121]:
fig = px.line(
    profil_horaire_zone,
    x="heure",
    y="consommation_moyenne_kwh",
    color="zone",
    markers=True,
    title="Profil horaire moyen par zone",
    labels={
        "heure": "Heure de la journée",
        "consommation_moyenne_kwh": "Consommation moyenne horaire (kWh)",
        "zone": "Zone"
    }
)

fig.show()

In [122]:
# Top 10 des relevés horaires les plus élevés

top_pics_horaires = (
    releves_horaires
    .sort_values(by="consommation_kwh", ascending=False)
    .head(10)
)

top_pics_horaires

,id_pdl,horodatage,consommation_kwh,zone,date,heure,jour_semaine,weekend
15714,PDL-000026,2025-09-16 18:00:00,63.06,Parc-Tertiaire,2025-09-16,18,1,False
15137,PDL-000026,2025-08-23 17:00:00,61.77,Parc-Tertiaire,2025-08-23,17,5,True
15233,PDL-000026,2025-08-27 17:00:00,61.68,Parc-Tertiaire,2025-08-27,17,2,False
15740,PDL-000026,2025-09-17 20:00:00,60.45,Parc-Tertiaire,2025-09-17,20,2,False
15356,PDL-000026,2025-09-01 20:00:00,60.08,Parc-Tertiaire,2025-09-01,20,0,False
15521,PDL-000026,2025-09-08 17:00:00,59.96,Parc-Tertiaire,2025-09-08,17,0,False
18762,PDL-000031,2025-08-24 18:00:00,59.39,Zone-Industrielle,2025-08-24,18,6,True
15296,PDL-000026,2025-08-30 08:00:00,58.44,Parc-Tertiaire,2025-08-30,8,5,True
18857,PDL-000031,2025-08-28 17:00:00,58.32,Zone-Industrielle,2025-08-28,17,3,False
15764,PDL-000026,2025-09-18 20:00:00,57.82,Parc-Tertiaire,2025-09-18,20,3,False


## 6. Analyse des réclamations client

Cette section analyse les réclamations clients afin d’apporter une première lecture orientée relation client.

L’objectif est d’identifier :
- le volume de réclamations ;
- les canaux de contact les plus utilisés ;
- l’évolution mensuelle des réclamations ;
- la satisfaction moyenne ;
- les signaux clients pouvant être rapprochés des incidents ou problèmes de consommation.

In [123]:
# Aperçu des réclamations clients

print("Dimensions réclamations :", reclamations.shape)
reclamations.head()

Dimensions réclamations : (3000, 9)


,id_reclamation,id_client,date,canal,texte,satisfaction,annee,mois,longueur_texte
0,REC-00001,CLI-00649,2025-12-07,email,Je n'ai pas donne mon accord pour la transmiss...,1,2025,12,120
1,REC-00002,CLI-00541,2025-12-02,espace_client,Je tiens a remercier le technicien intervenu e...,5,2025,12,91
2,REC-00003,CLI-00064,2025-01-24,courrier,Les microcoupures se multiplient depuis janvie...,2,2025,1,105
3,REC-00004,CLI-00570,2024-06-23,courrier,"Vous m'avez preleve deux fois ce mois-ci, je v...",2,2024,6,91
4,REC-00005,CLI-00088,2026-02-19,espace_client,"Je conteste ma facture de fevrier, le montant ...",2,2026,2,142


In [124]:
reclamations.columns.tolist()

['id_reclamation',
 'id_client',
 'date',
 'canal',
 'texte',
 'satisfaction',
 'annee',
 'mois',
 'longueur_texte']

In [125]:
# Indicateurs globaux des réclamations

kpis_reclamations = {
    "nb_reclamations": len(reclamations),
    "nb_clients_concernes": reclamations["id_client"].nunique(),
    "date_min": reclamations["date"].min(),
    "date_max": reclamations["date"].max(),
    "satisfaction_moyenne": round(reclamations["satisfaction"].mean(), 2),
    "satisfaction_mediane": round(reclamations["satisfaction"].median(), 2),
    "longueur_moyenne_texte": round(reclamations["longueur_texte"].mean(), 2),
}

kpis_reclamations_df = pd.DataFrame(
    list(kpis_reclamations.items()),
    columns=["indicateur", "valeur"]
)

kpis_reclamations_df

,indicateur,valeur
0,nb_reclamations,3000
1,nb_clients_concernes,687
2,date_min,2024-01-01 00:00:00
3,date_max,2026-05-29 00:00:00
4,satisfaction_moyenne,2.45
5,satisfaction_mediane,2.0
6,longueur_moyenne_texte,108.86


In [126]:
# Réclamations par canal

reclamations_par_canal = (
    reclamations
    .groupby("canal", as_index=False)
    .agg(
        nb_reclamations=("id_reclamation", "count"),
        satisfaction_moyenne=("satisfaction", "mean"),
        longueur_moyenne_texte=("longueur_texte", "mean")
    )
)

reclamations_par_canal["satisfaction_moyenne"] = reclamations_par_canal["satisfaction_moyenne"].round(2)
reclamations_par_canal["longueur_moyenne_texte"] = reclamations_par_canal["longueur_moyenne_texte"].round(2)

reclamations_par_canal = reclamations_par_canal.sort_values(
    by="nb_reclamations",
    ascending=False
)

reclamations_par_canal

,canal,nb_reclamations,satisfaction_moyenne,longueur_moyenne_texte
0,courrier,770,2.41,109.98
3,telephone,751,2.48,108.72
1,email,742,2.48,108.64
2,espace_client,737,2.42,108.07


In [127]:
fig = px.bar(
    reclamations_par_canal,
    x="canal",
    y="nb_reclamations",
    title="Nombre de réclamations par canal",
    labels={
        "canal": "Canal",
        "nb_reclamations": "Nombre de réclamations"
    }
)

fig.show()

In [128]:
fig = px.bar(
    reclamations_par_canal,
    x="canal",
    y="satisfaction_moyenne",
    title="Satisfaction moyenne par canal",
    labels={
        "canal": "Canal",
        "satisfaction_moyenne": "Satisfaction moyenne"
    }
)

fig.show()

L’analyse par canal permet d’identifier les canaux les plus sollicités et ceux où la satisfaction est la plus faible.

In [129]:
# Réclamations par mois

reclamations_mensuelles = (
    reclamations
    .groupby(["annee", "mois"], as_index=False)
    .agg(
        nb_reclamations=("id_reclamation", "count"),
        satisfaction_moyenne=("satisfaction", "mean"),
        nb_clients_concernes=("id_client", "nunique")
    )
)

reclamations_mensuelles["satisfaction_moyenne"] = reclamations_mensuelles["satisfaction_moyenne"].round(2)

reclamations_mensuelles["periode"] = pd.to_datetime(
    reclamations_mensuelles["annee"].astype(str) + "-"
    + reclamations_mensuelles["mois"].astype(str) + "-01"
)

reclamations_mensuelles.head()

,annee,mois,nb_reclamations,satisfaction_moyenne,nb_clients_concernes,periode
0,2024,1,106,2.44,101,2024-01-01
1,2024,2,88,2.44,83,2024-02-01
2,2024,3,97,2.64,93,2024-03-01
3,2024,4,109,2.65,104,2024-04-01
4,2024,5,104,2.64,99,2024-05-01


In [130]:
fig = px.line(
    reclamations_mensuelles,
    x="periode",
    y="nb_reclamations",
    markers=True,
    title="Évolution mensuelle du nombre de réclamations",
    labels={
        "periode": "Période",
        "nb_reclamations": "Nombre de réclamations"
    }
)

fig.show()

In [131]:
# Clients ayant le plus de réclamations

top_clients_reclamations = (
    reclamations
    .groupby("id_client", as_index=False)
    .agg(
        nb_reclamations=("id_reclamation", "count"),
        satisfaction_moyenne=("satisfaction", "mean"),
        longueur_moyenne_texte=("longueur_texte", "mean")
    )
)

top_clients_reclamations["satisfaction_moyenne"] = top_clients_reclamations["satisfaction_moyenne"].round(2)
top_clients_reclamations["longueur_moyenne_texte"] = top_clients_reclamations["longueur_moyenne_texte"].round(2)

top_clients_reclamations = top_clients_reclamations.sort_values(
    by="nb_reclamations",
    ascending=False
).head(10)

top_clients_reclamations

,id_client,nb_reclamations,satisfaction_moyenne,longueur_moyenne_texte
50,CLI-00051,12,3.33,106.58
161,CLI-00166,12,2.00,109.33
649,CLI-00661,11,2.09,120.00
119,CLI-00123,11,3.27,96.36
557,CLI-00567,11,3.00,95.64
408,CLI-00416,11,2.82,101.91
656,CLI-00669,11,3.00,102.55
95,CLI-00099,10,2.60,107.40
108,CLI-00112,10,2.70,119.30
89,CLI-00092,10,2.30,113.20


## Conclusion relation client

L’analyse des réclamations apporte une première lecture utile pour la Direction Relation Client.

Elle permet d’identifier :
- les canaux les plus utilisés ;
- les variations mensuelles du volume de réclamations ;
- les niveaux de satisfaction ;
- les clients ou périodes nécessitant une attention particulière.

# Synthèse des enseignements Data Analyst

Les analyses réalisées mettent en évidence plusieurs enseignements utiles pour Néovolt Grid+.

## 1. Consommation et saisonnalité

La consommation varie fortement selon les mois et les saisons. Les périodes froides sont associées à une consommation plus élevée, ce qui confirme l’importance de la météo dans l’anticipation des pics.

## 2. Profils de consommation

Les consommations diffèrent selon la zone, le type de client, le segment commercial et le type de chauffage. Ces dimensions doivent être intégrées dans les tableaux de bord afin d’éviter une lecture trop globale.

## 3. Influence de la météo

Les degrés-jour de chauffage montrent une relation positive avec la consommation. Plus les journées sont froides, plus la consommation tend à augmenter. La météo est donc un facteur clé pour l’exploitation réseau et la direction financière.

## 4. Incidents réseau

Les incidents peuvent être analysés par zone, type et cause. Cette lecture aide à identifier les zones les plus sensibles et à orienter la maintenance préventive. Le croisement consommation/incidents reste descriptif et ne permet pas, à lui seul, d’établir une causalité.

## 5. Profils horaires

L’analyse horaire met en évidence des heures de pointe et des différences entre jours ouvrés et week-ends. Même si l’échantillon horaire est limité, il montre l’intérêt d’une surveillance fine des courbes de charge.

## 6. Relation client

Les réclamations permettent d’enrichir la vision métier avec une lecture orientée satisfaction client. Elles peuvent être croisées avec les incidents, les zones et les périodes de forte consommation dans un tableau de bord relation client.

## Recommandations métier provisoires

- Mettre en place une vue de pilotage mensuelle et saisonnière de la consommation.
- Surveiller les zones et segments fortement consommateurs.
- Intégrer la météo dans les analyses de prévision et les tableaux de bord.
- Suivre les incidents par zone, type et cause pour prioriser les actions réseau.
- Exploiter les profils horaires pour identifier les heures de pointe.
- Ajouter une vue relation client pour suivre réclamations et satisfaction.
- Documenter systématiquement les limites de qualité des données.

## Limites

- Le fichier horaire ne couvre qu’un échantillon limité de points de livraison.
- Les analyses sont descriptives et ne prouvent pas de causalité.
- Les données nettoyées reposent sur des règles d’imputation documentées.
- Les résultats devront être consolidés dans Power BI et complétés par les modèles du volet Data Scientist.

## Dimensions transverses

Les analyses sont réalisées avec prudence :
- les données personnelles sont utilisées sous forme agrégée autant que possible ;
- les limites de qualité sont documentées ;
- les résultats ne doivent pas conduire à des décisions automatiques défavorables aux clients ;
- les traitements restent simples et reproductibles afin de limiter la complexité et l’empreinte de calcul.